# 07 · PHM 2016 CMP — 실데이터 회귀 VM

**진짜 반도체 Virtual Metrology**: CMP(화학적기계연마) 공정의 in-situ 센서로 **평균 제거율(MRR)**을 예측. 05번 합성 데이터를 실데이터로 대체.

> 데이터: PHM Society 2016 Data Challenge (무료, 등록). `data/raw/phm/`에 배치.

## 0. 준비
PHM 2016 CMP 파일을 `data/raw/phm/`에 두고 컬럼명을 로더 인자에 맞춘다.
(센서 시계열 → wafer/stage 단위 mean/std/min/max 집계)

In [ ]:
import sys; sys.path.append("..")
from src.datasets.phm_cmp import load_phm_cmp
# 실제 컬럼명에 맞게 조정
X, y = load_phm_cmp(id_cols=("WAFER_ID","STAGE"), target="AVG_REMOVAL_RATE")
print("X:", X.shape, "| MRR range:", round(float(y.min()),2), "~", round(float(y.max()),2))

## 1. 회귀 모델 비교 (PLS·RF·GBM) — RMSE·R²
05번과 **동일한 회귀 파이프라인** 재사용.

In [ ]:
from src.regression import evaluate
evaluate(X, y, cv=5)

## 2. Parity plot — 예측 vs 실제 MRR

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_predict
from src.regression import vm_pipeline
pred = cross_val_predict(vm_pipeline(RandomForestRegressor(n_estimators=400, n_jobs=-1, random_state=42)), X, y, cv=5)
plt.scatter(y, pred, s=8, alpha=.4, color="#1A4D8F")
lim=[y.min(), y.max()]; plt.plot(lim, lim, "r--")
plt.xlabel("true MRR"); plt.ylabel("predicted MRR"); plt.title("PHM CMP — VM parity"); plt.show()

> **포인트**: 합성→실제 CMP로 교체 = 정통 반도체 VM 회귀 증명. 같은 코드, 실데이터.